# 管理租户状态和温度

<img src = "https://weaviate.io/assets/images/storage-tiers-42d32dfd416335c0a1ff0ffb257d978b.jpg">

存储资源分为不同的层级。每层级的性能特征和成本各不相同：

\begin{array}{|l|l|l|l|}
\hline \text { 层级 } & \text { 地点 } & \text { 速度 } & \text { 成本 } \\
\hline \text { 热的 } & \text { 内存 } & \text { 最快访问 } & \text { 最昂贵的 } \\
\hline \text { 温暖的 } & \text { 磁盘 } & \text { 中速 } & \text { 中等价格 } \\
\hline \text { 寒冷的 } & \text { 云存储 } & \text { 最慢访问 } & \text { 最便宜 } \\
\hline
\end{array}


热存储和冷存储之间的价格差异很大。云存储比 RAM 便宜几个数量级。

在多租户集合中，您可以更改租户状态（Active、Inactive、Offloaded）以在存储层之间移动数据。这允许在成本、资源可用性和就绪性之间进行精细的权衡。

## 租户状态

租户状态有三种：Active、Inactive和Offloaded。

| 租户状态 | CRUD 和查询 | 向量索引 | 倒排索引 | 对象数据 | 激活时间 | 描述 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| Active（默认） | 是的 | 热温 | 温暖的 | 温暖的 | 没有任何 | 租户可以使用 |
| Inactive | 不 | 温暖的 | 温暖的 | 温暖的 | 快速地 | 租户在本地存储但不可用 |
| Offloaded | 不 | 寒冷的 | 寒冷的 | 寒冷的 | 慢的 | 租户存储在云存储中，不可使用 |

租户Active可用于查询和 CRUD 操作。根据向量索引类型，它使用热资源或温资源。

租户的对象数据和倒排索引都存储在磁盘上，占用warm资源。

<img src="https://weaviate.io/assets/images/active-tenants-658dde3f9e05fa5ebdc24187e0f15b2c.jpg">

租户Inactive不可用于查询或 CRUD 操作。

租户的对象数据、向量索引和倒排索引存储在磁盘上，占用warm资源。与占用hot资源的活跃租户相比，这可以降低 Weaviate 的内存需求。

由于租户存储在本地，因此可以快速激活非活动租户。


<img src="https://weaviate.io/assets/images/inactive-tenants-79dcc442d93ddac3fdca67df83fcbee4.jpg">

租户offloaded不可用于查询或 CRUD 操作。

租户的对象数据、向量索引和倒排索引存储在云端，占用cold资源。由于租户存储在远程，因此激活已卸载的租户时会存在延迟。

<img src="https://weaviate.io/assets/images/offloaded-tenants-762fbe0ae29452b2a8835fd445de12ed.jpg">

## 激活租户

要从磁盘激活租户，或从云INACTIVE加载并激活租户，请调用：OFFLOADED


In [ ]:
from weaviate.classes.tenants import Tenant, TenantActivityStatus

multi_collection = client.collections.get("MultiTenancyCollection")
multi_collection.tenants.update(tenants=[
    Tenant(
        name="tenantA",
        activity_status=TenantActivityStatus.ACTIVE
    )
])

## 停用租户

要停用ACTIVE租户，或从云端加载OFFLOADED租户（而不激活它），请调用：

In [ ]:
from weaviate.classes.tenants import Tenant, TenantActivityStatus

multi_collection = client.collections.get("MultiTenancyCollection")
multi_collection.tenants.update(tenants=[
    Tenant(
        name="tenantA",
        activity_status=TenantActivityStatus.INACTIVE
    )
])

## 卸载租户
要将ACTIVE或INACTIVE租户卸载到云，请致电：

In [ ]:
from weaviate.classes.tenants import Tenant, TenantActivityStatus

multi_collection = client.collections.get("MultiTenancyCollection")
multi_collection.tenants.update(tenants=[
    Tenant(
        name="tenantA",
        activity_status=TenantActivityStatus.OFFLOADED
    )
])